In [1]:
import numpy as np 
import pandas as pd
import re 
import datetime
import traceback as tb
import sys
import math
import unicodedata
from datetime import time

In [3]:
df_ctd = pd.read_excel(r'C:\Users\felipe.abarzua\Desktop\workspace\DATA_AMBIENTALES\Proyecto Seguimiento Ambiental\CTD_2019-2024\6_BD_CTD_SEGUIMIENTO_2024.xlsx')

In [4]:
df_ctd

,Nombre del Proyecto:,"ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO, 2024 -2025",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,Codigo del Proyecto:,656-171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Etapa Proyecto:,Objetivo 1. Actividad 2. Muestreos y Análisis ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jefe Proyecto,Johana Ojeda Palma,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Institución:,Instituto de Fomento Pesquero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sectores:,"Estuario de Reloncaví, Seno de Reloncaví, Golf...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6571,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,247,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6572,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,248,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6573,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,249,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6574,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,250,251,...,11.8915,1.8955,19.188,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6


In [13]:
def limpieza_ctd(df):

    # Creamos una copia del dataset
    df_ctd_copy = df.copy()

    # Eliminamos las filas innecesarias
    df_ctd_copy = df_ctd_copy.iloc[22:  , : ]

    # Dejamos la primera fila como columnas
    df_ctd_copy.columns = df_ctd_copy.iloc[0]

    # Eliminamos la primera fila 
    df_ctd_copy = df_ctd_copy.iloc[1: , :]

    #Reiniciamos los indices
    df_ctd_copy = df_ctd_copy.reset_index(drop = True)

    #Agregamos la columna ID 
    df_ctd_copy.insert(0 , 'ID' , df_ctd_copy.index + 1)

    # Modificamos las columnas de estación ya que existen dos de ellas
    df_ctd_copy.columns.values[8] = 'ESTACION_1'
    df_ctd_copy.columns.values[24] = 'ESTACION_2'

    #Creamos la lista de columnas 
    columnas = df_ctd_copy.columns.to_list()

    #Eliminamos la estacion_2 ya que es innecesaria
    df_ctd_copy.drop(columns=['ESTACION_2'], inplace=True)

    # Renombramos la ESTACION_1 por ESTACION
    df_ctd_copy.columns.values[8] = 'ESTACION'

    #Creamos la lista de columnas 
    columnas = df_ctd_copy.columns.to_list()
    
    #Creamos las columnas fijas y variables
    columnas_fijas = ['ID', 'NOMB_PROY', 'COD_PROY', 'FECHA_INI', 'FECHA_TER', 'EQUIPO', 'LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT' , 'HORA_INICIO' , 'COD_REG' , 'SECTOR']
    columnas_variables = [ 'TEMPERATUR', 'OXIG_ml/L', 'OXIG_mg/L', 'OXIG_%sat', 'OX_umol/kg','SALINIDAD', 'DENSIDAD', 'CLOROFILA', 'FEOPIGMEN' , 'D_SECCHI']

    
    #Realizamos un melt para convertir las columnas_variables en filas 
    df_ctd_copy = pd.melt(df_ctd_copy , id_vars= columnas_fijas , value_vars= columnas_variables , var_name= 'VARIABLE' , value_name='VALOR')


        #Vamos a reemplazar los valores e HORA_INICIO ya que están mal escritos

    def normalizar_hora_string(hora_str):

            
            #Normaliza una cadena de tiempo para asegurar que los minutos tengan dos dígitos.
            #Ej: '12:5' se convierte en '12:05'.
            
        if pd.isna(hora_str): # Maneja posibles valores NaN/nulos si los hubiera
            return hora_str

        partes = str(hora_str).split(':')
        if len(partes) != 2:
                # Manejar casos donde el formato no es 'HH:MM' (ej. ya está mal, o es un dato inesperado)
                # Puedes decidir si quieres levantar un error, devolver el original, o un valor específico.
                # Por simplicidad, devolveremos el original si el formato no es el esperado de dos partes.
            return hora_str

        horas = partes[0]
        minutos = partes[1]

        if len(minutos) == 1:
            minutos = '0' + minutos # Añadir el cero delante si es un solo dígito

            return f"{horas}:{minutos}"
        
        #Aplicamos la funcion de hora a la columnas HORA_INICIO
        df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(normalizar_hora_string)

        # Creamos una funcion para convertir la columna de HORA_INICIO a datetime para luego Crear una columna de fecha y hora 
        def convertir_hora(time_value):
            if isinstance(time_value, str): 
                # Separamos el ":" del texto y lo convertimos a entero y obtenemos dos variables
                horas, minutos = map(int, time_value.split(':'))
                #Retornamos los valores de Horas y minutos
                return datetime.time(horas, minutos)
            else:
                # En caso que ya es datetime.time se deja como esta
                return time_value


        df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(convertir_hora)


         #Cambiamos los tipos de FECHA_INI y FECHA_FIN a datetime

        df_ctd_copy['FECHA_INI'] = pd.to_datetime(df_ctd_copy['FECHA_INI'])
        df_ctd_copy['FECHA_TER'] = pd.to_datetime(df_ctd_copy['FECHA_TER'])

        #FECHA INI CONVERTIDA A STRING 
        df_ctd_copy['FECHA_INI_STR'] =  df_ctd_copy['FECHA_INI'].dt.strftime("%d/%m/%Y")
        df_ctd_copy['FECHA_TERMINO'] = df_ctd_copy['FECHA_TER'].dt.strftime("%d/%m/%Y")

        
        #Creamos una funcion lambda para combinar FECHA_INI con la HORA_INI
        df_ctd_copy['FECHA_INICIO'] = df_ctd_copy.apply(
        lambda row: datetime.datetime.strptime(row['FECHA_INI_STR'], "%d/%m/%Y").replace(
            hour=row['HORA_INICIO'].hour,
            minute=row['HORA_INICIO'].minute,
            second=row['HORA_INICIO'].second
            ),
            axis=1
        )

        # Modificamos el formato de FECHA_INICIO 
        df_ctd_copy['FECHA_INICIO'] = df_ctd_copy['FECHA_INICIO'].dt.strftime("%d/%m/%Y %H:%M:%S")

        # Eliminamos las columnas innecesarias

        df_ctd_copy = df_ctd_copy[['ID', 'NOMB_PROY', 'COD_PROY', 'EQUIPO','LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT', 'COD_REG', 'SECTOR', 'VARIABLE', 'VALOR',
                                'FECHA_TERMINO', 'FECHA_INICIO']]

        # Cambiamos las variables
        df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'D_SECCHI' , 'DISCO SECCHI' , df_ctd_copy['VARIABLE'] )
        df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'TEMPERATUR' , 'TEMPERATURA' , df_ctd_copy['VARIABLE'] )
        
        # Dejamos las variable en mayuscula
        df_ctd_copy['VARIABLE'] = df_ctd_copy['VARIABLE'].str.upper()


    return df_ctd_copy , columnas




In [14]:
df_ctd_copy , columnas = limpieza_ctd(df_ctd)

AttributeError: 'DataFrame' object has no attribute 'dtype'